# Modelo de Ablación Fischer–Rajpal

Este cuaderno implementa el **modelo intermedio** del proyecto, usando:

- **Arquitectura tipo Fischer & Krauss**: una sola capa LSTM con 25 unidades ocultas.
- **Entrada tipo Rajpal**: secuencias multivariadas de 20 días con 5 variables OHLCV transformadas a porcentaje de cambio.
- **Salida binaria**: clase 1 si el retorno futuro de la acción es mayor o igual a la mediana transversal del día; clase 0 si está por debajo.

La idea metodológica es aislar una pregunta específica:

> ¿Qué ocurre si mantenemos una LSTM simple tipo Fischer, pero reemplazamos la entrada univariada de retornos por la representación multivariada OHLCV tipo Rajpal?

Este modelo **no usa atención, conexión residual, LayerNormalization, PReLU ni cabeza de regresión**, porque esos elementos corresponden al modelo final tipo Rajpal y mezclarían los efectos de arquitectura con los efectos de representación de datos.


## 1. Importar librerías y fijar semilla

Se fija una semilla para mejorar la reproducibilidad. En redes neuronales puede haber pequeñas variaciones entre ejecuciones por diferencias de hardware, GPU o versiones de TensorFlow.


In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.initializers import GlorotUniform, Orthogonal, Zeros

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    log_loss,
    confusion_matrix,
    classification_report
)

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

print("TensorFlow:", tf.__version__)


## 2. Configuración de rutas

El cuaderno espera encontrar el dataset en:

```text
data/financial_dataset.npz
```

También incluye rutas alternativas por si se está ejecutando directamente en una carpeta con el archivo `.npz`.

El archivo debe contener como mínimo:

```text
X
y_classification
sequence_dates
```

Idealmente también puede contener:

```text
y_regression
sequence_tickers
```


In [ ]:
BASE_DIR = Path.cwd()

DATA_CANDIDATES = [
    BASE_DIR / "data" / "financial_dataset.npz",
    BASE_DIR / "financial_dataset.npz",
    BASE_DIR / "financial_dataset(1).npz",
    Path("/mnt/data/financial_dataset(1).npz")
]

DATA_PATH = None
for candidate in DATA_CANDIDATES:
    if candidate.exists():
        DATA_PATH = candidate
        break

if DATA_PATH is None:
    raise FileNotFoundError(
        "No se encontró financial_dataset.npz. "
        "Ubica el archivo en data/financial_dataset.npz o en la misma carpeta del cuaderno."
    )

MODELS_DIR = BASE_DIR / "models"
REPORTS_DIR = BASE_DIR / "reports"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODELS_DIR / "fischer_rajpal_ablation_lstm.keras"
METRICS_PATH = REPORTS_DIR / "fischer_rajpal_ablation_metrics.csv"
HISTORY_PATH = REPORTS_DIR / "fischer_rajpal_ablation_history.csv"

print("DATA_PATH:", DATA_PATH)
print("MODEL_PATH:", MODEL_PATH)


## 3. Cargar el dataset

Este dataset corresponde a la entrada tipo Rajpal:

```text
X.shape = (n_muestras, 20, 5)
```

donde cada muestra representa una secuencia de 20 días y cada día tiene 5 variables:

```text
open_pct_change, high_pct_change, low_pct_change, close_pct_change, volume_pct_change
```


In [ ]:
def load_dataset(data_path):
    data = np.load(data_path, allow_pickle=True)

    print("Llaves disponibles en el .npz:")
    print(data.files)

    required_keys = ["X", "y_classification", "sequence_dates"]
    missing = [key for key in required_keys if key not in data.files]
    if missing:
        raise KeyError(f"Faltan llaves obligatorias en el .npz: {missing}")

    X = data["X"].astype("float32")
    y = data["y_classification"].astype("int64")
    sequence_dates = data["sequence_dates"]

    sequence_tickers = data["sequence_tickers"] if "sequence_tickers" in data.files else None
    y_regression = data["y_regression"] if "y_regression" in data.files else None

    return X, y, sequence_dates, sequence_tickers, y_regression


X, y, sequence_dates, sequence_tickers, y_regression = load_dataset(DATA_PATH)

print("\nDimensiones:")
print("X:", X.shape)
print("y:", y.shape)
print("sequence_dates:", sequence_dates.shape)

if sequence_tickers is not None:
    print("sequence_tickers:", sequence_tickers.shape)

if y_regression is not None:
    print("y_regression:", y_regression.shape)


## 4. Verificaciones rápidas del dataset

Estas verificaciones sirven para confirmar que el dataset **ya no está usando precios crudos**.

Se espera que las variables de precio estén en cambios porcentuales y que la etiqueta binaria esté aproximadamente balanceada, porque se construyó con la mediana transversal de cada fecha.


In [ ]:
feature_names = [
    "open_pct_change",
    "high_pct_change",
    "low_pct_change",
    "close_pct_change",
    "volume_pct_change"
]

print("NaN en X:", np.isnan(X).sum())
print("Inf en X:", np.isinf(X).sum())
print("NaN en y:", np.isnan(y).sum() if np.issubdtype(y.dtype, np.floating) else "No aplica")

print("\nRango global de X:")
print("min:", np.nanmin(X))
print("max:", np.nanmax(X))
print("mean:", np.nanmean(X))

X_flat = X.reshape(-1, X.shape[-1])
summary = pd.DataFrame(X_flat, columns=feature_names).describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
)

summary


In [ ]:
class_counts = pd.Series(y).value_counts().sort_index()
class_distribution = pd.DataFrame({
    "count": class_counts,
    "percentage": class_counts / len(y) * 100
})

class_distribution


## 5. División temporal train / validation / test

Se usa una división cronológica sencilla:

- **Train**: fechas antes de 2017-01-01.
- **Validation**: fechas desde 2017-01-01 hasta antes de 2018-01-01.
- **Test**: fechas desde 2018-01-01 en adelante.

Esto permite mantener una evaluación out-of-sample sin mezclar datos futuros en el entrenamiento.


In [ ]:
def split_data(X, y, sequence_dates):
    sequence_dates = sequence_dates.astype("datetime64[D]")

    train_mask = sequence_dates < np.datetime64("2017-01-01")

    val_mask = (
        (sequence_dates >= np.datetime64("2017-01-01")) &
        (sequence_dates < np.datetime64("2018-01-01"))
    )

    test_mask = sequence_dates >= np.datetime64("2018-01-01")

    splits = {
        "train": (X[train_mask], y[train_mask], sequence_dates[train_mask]),
        "validation": (X[val_mask], y[val_mask], sequence_dates[val_mask]),
        "test": (X[test_mask], y[test_mask], sequence_dates[test_mask])
    }

    return splits


splits = split_data(X, y, sequence_dates)

X_train, y_train, dates_train = splits["train"]
X_val, y_val, dates_val = splits["validation"]
X_test, y_test, dates_test = splits["test"]

for split_name, (X_split, y_split, dates_split) in splits.items():
    print("\n" + "=" * 50)
    print(split_name.upper())
    print("=" * 50)
    print("X:", X_split.shape)
    print("y:", y_split.shape)
    print("Fecha mínima:", dates_split.min())
    print("Fecha máxima:", dates_split.max())
    print(pd.Series(y_split).value_counts(normalize=True).sort_index())


## 6. Modelo de Ablación Fischer–Rajpal

Este es el punto central del cuaderno.

La arquitectura mantiene la lógica de Fischer:

```text
Input
↓
LSTM con 25 unidades ocultas
↓
Dense(2, softmax)
```

Pero la entrada ya no es una secuencia univariada de 240 retornos. Ahora es:

```text
Input shape = (20, 5)
```

que corresponde a la representación multivariada OHLCV tipo Rajpal.


In [ ]:
def build_fischer_rajpal_ablation_model(seq_len=20, n_features=5, n_classes=2):
    inputs = Input(
        shape=(seq_len, n_features),
        name="ohlcv_pct_change_sequence"
    )

    x = LSTM(
        units=25,
        return_sequences=False,
        activation="tanh",
        recurrent_activation="sigmoid",
        dropout=0.1,
        recurrent_dropout=0.1,
        kernel_initializer=GlorotUniform(seed=SEED),
        recurrent_initializer=Orthogonal(seed=SEED),
        bias_initializer=Zeros(),
        name="fischer_lstm_25"
    )(inputs)

    outputs = Dense(
        n_classes,
        activation="softmax",
        name="classification_output"
    )(x)

    model = Model(
        inputs=inputs,
        outputs=outputs,
        name="Fischer_Rajpal_Input_Ablation_Model"
    )

    return model


model = build_fischer_rajpal_ablation_model(
    seq_len=X.shape[1],
    n_features=X.shape[2],
    n_classes=2
)

model.summary()


## 7. Compilación

Se usa:

- **Optimizador**: RMSprop, consistente con el enfoque de Fischer para redes recurrentes.
- **Función de pérdida**: Sparse Categorical Crossentropy.
- **Métrica durante entrenamiento**: accuracy.

Las métricas más completas, como precision, recall, F1 y AUC, se calculan después de entrenar.


In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.RMSprop(learning_rate=1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")
    ]
)


## 8. Entrenamiento

Se entrena con `shuffle=False` para preservar el orden temporal dentro de los datos.

El checkpoint guarda el mejor modelo según `val_accuracy`.


In [ ]:
early_stopping = EarlyStopping(
    monitor="val_accuracy",
    patience=10,
    restore_best_weights=True,
    mode="max",
    verbose=1
)

checkpoint = ModelCheckpoint(
    filepath=MODEL_PATH,
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1
)

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    callbacks=[
        early_stopping,
        checkpoint
    ],
    epochs=20,
    batch_size=512,
    shuffle=False,
    verbose=1
)

history_df = pd.DataFrame(history.history)
history_df.to_csv(HISTORY_PATH, index=False)

print("Historial guardado en:", HISTORY_PATH)


## 9. Curvas de entrenamiento

Estas curvas permiten revisar si el modelo está aprendiendo, si se estanca en 50%, o si hay sobreajuste.


In [ ]:
history_df = pd.DataFrame(history.history)

plt.figure(figsize=(8, 5))
plt.plot(history_df["loss"], label="train_loss")
plt.plot(history_df["val_loss"], label="val_loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Curva de pérdida")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history_df["accuracy"], label="train_accuracy")
plt.plot(history_df["val_accuracy"], label="val_accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Curva de accuracy")
plt.legend()
plt.grid(True)
plt.show()


## 10. Evaluación del modelo

Aquí se calculan métricas comparables con otros modelos:

- Accuracy
- Precision
- Recall
- F1-score
- ROC AUC
- Log loss
- Matriz de confusión

La probabilidad de clase 1 se toma de la segunda columna del softmax.


In [ ]:
def evaluate_model(model, X_split, y_split, split_name):
    y_proba = model.predict(X_split, batch_size=1024, verbose=0)
    y_pred = np.argmax(y_proba, axis=1)

    proba_class_1 = y_proba[:, 1]

    metrics = {
        "split": split_name,
        "accuracy": accuracy_score(y_split, y_pred),
        "precision": precision_score(y_split, y_pred, zero_division=0),
        "recall": recall_score(y_split, y_pred, zero_division=0),
        "f1_score": f1_score(y_split, y_pred, zero_division=0),
        "log_loss": log_loss(y_split, y_proba)
    }

    try:
        metrics["roc_auc"] = roc_auc_score(y_split, proba_class_1)
    except ValueError:
        metrics["roc_auc"] = np.nan

    cm = confusion_matrix(y_split, y_pred)

    print("\n" + "=" * 60)
    print(split_name.upper())
    print("=" * 60)
    print(pd.Series(metrics))
    print("\nClassification report:")
    print(classification_report(y_split, y_pred, zero_division=0))
    print("\nConfusion matrix:")
    print(cm)

    return metrics, cm, y_pred, y_proba


train_metrics, train_cm, train_pred, train_proba = evaluate_model(
    model, X_train, y_train, "train"
)

val_metrics, val_cm, val_pred, val_proba = evaluate_model(
    model, X_val, y_val, "validation"
)

test_metrics, test_cm, test_pred, test_proba = evaluate_model(
    model, X_test, y_test, "test"
)

metrics_df = pd.DataFrame([
    train_metrics,
    val_metrics,
    test_metrics
])

metrics_df.to_csv(METRICS_PATH, index=False)

print("\nMétricas guardadas en:", METRICS_PATH)
metrics_df


## 11. Matriz de confusión en validación y prueba

La matriz de confusión permite detectar colapsos del modelo, por ejemplo cuando predice todo como clase 0 o todo como clase 1.


In [ ]:
def plot_confusion_matrix(cm, title):
    plt.figure(figsize=(5, 4))
    plt.imshow(cm)
    plt.title(title)
    plt.xlabel("Predicted label")
    plt.ylabel("True label")
    plt.xticks([0, 1], ["Class 0", "Class 1"])
    plt.yticks([0, 1], ["Class 0", "Class 1"])

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, cm[i, j], ha="center", va="center")

    plt.colorbar()
    plt.tight_layout()
    plt.show()


plot_confusion_matrix(val_cm, "Confusion matrix - validation")
plot_confusion_matrix(test_cm, "Confusion matrix - test")


## 12. Guardar predicciones de validación y prueba

Esto facilita comparar este modelo contra otros modelos del proyecto.


In [ ]:
validation_predictions = pd.DataFrame({
    "date": dates_val.astype(str),
    "y_true": y_val,
    "y_pred": val_pred,
    "proba_class_0": val_proba[:, 0],
    "proba_class_1": val_proba[:, 1]
})

test_predictions = pd.DataFrame({
    "date": dates_test.astype(str),
    "y_true": y_test,
    "y_pred": test_pred,
    "proba_class_0": test_proba[:, 0],
    "proba_class_1": test_proba[:, 1]
})

if sequence_tickers is not None:
    all_dates = sequence_dates.astype("datetime64[D]")
    val_mask = (
        (all_dates >= np.datetime64("2017-01-01")) &
        (all_dates < np.datetime64("2018-01-01"))
    )
    test_mask = all_dates >= np.datetime64("2018-01-01")

    validation_predictions.insert(1, "ticker", sequence_tickers[val_mask])
    test_predictions.insert(1, "ticker", sequence_tickers[test_mask])

VALIDATION_PRED_PATH = REPORTS_DIR / "fischer_rajpal_validation_predictions.csv"
TEST_PRED_PATH = REPORTS_DIR / "fischer_rajpal_test_predictions.csv"

validation_predictions.to_csv(VALIDATION_PRED_PATH, index=False)
test_predictions.to_csv(TEST_PRED_PATH, index=False)

print("Predicciones de validación:", VALIDATION_PRED_PATH)
print("Predicciones de prueba:", TEST_PRED_PATH)

test_predictions.head()


## 13. Interpretación esperada

Este modelo debe compararse contra:

1. **Fischer puro**: LSTM simple con entrada univariada de retornos.
2. **Fischer–Rajpal ablation**: este cuaderno, LSTM simple con entrada OHLCV multivariada.
3. **Rajpal-based classifier**: LSTM con atención, residual, normalización y mayor profundidad.

La comparación correcta no es solo mirar si el accuracy sube, sino revisar también:

- Si la matriz de confusión está balanceada.
- Si el modelo predice ambas clases.
- Si el F1-score mejora.
- Si el AUC es superior a 0.5.
- Si los resultados de test se mantienen cerca de validación.
- Si no aparecen pérdidas `NaN`.

Si este modelo supera al Fischer puro, pero queda por debajo del Rajpal completo, eso apoyaría la idea de que **la representación OHLCV aporta información adicional**, pero que **la arquitectura más compleja también contribuye**.
